# 04_gee_refinement_flow.ipynb

利用 GEE 高分影像对争议热点进行真值判定。集成 Sentinel-2 (Cloud Score+) 与 Landsat。

In [ ]:
import sys, os
sys.path.append("../src")
import ee
import geemap
from wetland_analysis.data.satellite_fetcher import GEEFetcher
from wetland_analysis.analysis.consensus import calculate_weighted_consensus

# 初始化 GEE (需环境变量 GEE_PROJECT_ID)
fetcher = GEEFetcher()
os.environ['LOCALTILESERVER_CLIENT_PREFIX'] = 'proxy/{port}'

## 1. 区域与影像获取

针对选定的争议样区，获取 2020-2025 年间质量最高的合成影像。

In [ ]:
# 定义样区点 (亚马逊样点)
point = ee.Geometry.Point([-60.05, -3.15])
roi = point.buffer(10000)

# 获取 S2 (Cloud Score+ 掩码处理后)
s2_median = fetcher.get_sentinel2_image(roi, '2023-01-01', '2023-12-31')

print("Cloud-masked image fetched.")

## 2. 集成共识图与遥感影像对比

左侧展示原始遥感影像，右侧展示我们的 30m 集成共识结果。

In [ ]:
m = geemap.Map(center=[-3.15, -60.05], zoom=13)

# 1. 影像图层 (RGB)
vis_params = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000, 'gamma': 1.4}
m.addLayer(s2_median, vis_params, 'Sentinel-2 (2023)')

# 2. 共识图层 (Wetland Mask)
# 这里需要加载您本地保存的 consensus.tif 或实时计算结果
# consensus_layer = geemap.xarray_tile(consensus_ds, colormap='Blues', name='30m Consensus')

m.add_layer_control()
m